# Phase 4 — Fairness Baseline

Three parts, all against LightGBM's out-of-fold calibrated PD (Phase 3 winner) at its profit-optimal threshold (0.220):
1. DPD / EOD / Equalized Odds on real attributes (CODE_GENDER, NAME_FAMILY_STATUS, age_band)
2. Detection check on the synthetic overlay's planted CODE_GENDER label bias
3. Fairness-accuracy trade-off report across mitigation strategies

All three confirmed via `/leakage-check`.

## 1. Real-attribute fairness metrics

Runs `src/fairness/fairness_metrics.py`. Small subgroups (n < 500, e.g. CODE_GENDER's 4-row `XNA` and family status's 2-row `Unknown`) are excluded from the DPD/EOD/EqualizedOddsDiff spread calculation, not just flagged -- an earlier version only warned about them without excluding them, which let a handful of by-chance rows dominate the headline number (CODE_GENDER's DPD looked like 0.031 purely from XNA's 4 rows all being approved; the corrected, reliable number is 0.001).

In [1]:
%run ../src/fairness/fairness_metrics.py


=== CODE_GENDER ===
Approval rate by group:
              rate       n
CODE_GENDER               
F            0.970  202448
M            0.969  105059
XNA          1.000       4
DPD=0.0010  EOD=0.0001  EqualizedOddsDiff=0.0159

=== NAME_FAMILY_STATUS ===
Approval rate by group:
                        rate       n
NAME_FAMILY_STATUS                  
Civil marriage        0.9618   29775
Married               0.9710  196432
Separated             0.9763   19770
Single / not married  0.9633   45444
Unknown               1.0000       2
Widow                 0.9781   16088
DPD=0.0162  EOD=0.0115  EqualizedOddsDiff=0.0328

=== age_band ===
Approval rate by group:
            rate      n
age_band               
18-24     0.9330  12233
25-34     0.9590  72429
35-44     0.9723  84261
45-54     0.9749  70190
55-64     0.9774  60522
65+       0.9895   7876
DPD=0.0565  EOD=0.0472  EqualizedOddsDiff=0.0660

Saved: D:\finance\credit-risk-fyp\data\processed\phase4_real_fairness_metrics.csv


In [2]:
result

,DPD,EOD,EqualizedOddsDiff
attribute,,,
CODE_GENDER,0.000989,0.000104,0.015886
NAME_FAMILY_STATUS,0.016211,0.011527,0.032785
age_band,0.056493,0.047180,0.065980


## 2. Synthetic overlay: is the planted CODE_GENDER bias detectable?

Runs `src/fairness/synthetic_bias_detection.py`. The naive cross-period comparison (P0 clean vs P1-P4 biased) is confounded by simultaneous covariate-drift resampling and is kept only for context -- the real evidence is the **isolated effect** table: same population, same model decisions, per period, comparing the fairness metric against `TARGET_original` (pre-flip) vs `TARGET` (post-flip). This holds the covariate-drift-driven population composition fixed and isolates exactly what the label-flip mechanism contributes.

In [3]:
%run ../src/fairness/synthetic_bias_detection.py

=== Cross-period comparison (confounded by covariate drift -- context only) ===
           DPD     EOD  EqualizedOddsDiff
period                                   
P0      0.0011  0.0020             0.0215
P1      0.0016  0.0012             0.0127
P2      0.0006  0.0054             0.0372
P3      0.0038  0.0038             0.0184
P4      0.0087  0.0053             0.0084

=== Isolated bias effect: same population, TARGET_original vs TARGET ===
        DPD_true  DPD_biased  F_FPR_true  F_FPR_biased  F_FPR_inflation  EqualizedOddsDiff_true  EqualizedOddsDiff_biased
period                                                                                                                   
P1        0.0016      0.0016      0.8738        0.9140           0.0402                  0.0275                    0.0127
P2        0.0006      0.0006      0.8806        0.9126           0.0319                  0.0053                    0.0372
P3        0.0038      0.0038      0.8927        0.9175          

In [4]:
print("Cross-period (confounded, context only):")
display(cross_period.round(4))
print("\nIsolated effect (the actual evidence):")
display(isolated.round(4))

Cross-period (confounded, context only):


,DPD,EOD,EqualizedOddsDiff
period,,,
P0,0.0011,0.0020,0.0215
P1,0.0016,0.0012,0.0127
P2,0.0006,0.0054,0.0372
P3,0.0038,0.0038,0.0184
P4,0.0087,0.0053,0.0084



Isolated effect (the actual evidence):


,DPD_true,DPD_biased,F_FPR_true,F_FPR_biased,F_FPR_inflation,EqualizedOddsDiff_true,EqualizedOddsDiff_biased
period,,,,,,,
P1,0.0016,0.0016,0.8738,0.9140,0.0402,0.0275,0.0127
P2,0.0006,0.0006,0.8806,0.9126,0.0319,0.0053,0.0372
P3,0.0038,0.0038,0.8927,0.9175,0.0247,0.0064,0.0184
P4,0.0087,0.0087,0.8533,0.8850,0.0317,0.0401,0.0084


**Result**: DPD is identical whether measured against `TARGET_original` or `TARGET` in every period -- expected, since DPD never reads TARGET and is structurally blind to a label-only bias. F's false-positive rate (approved | labeled-bad) inflates by 2.5-4.0 percentage points in every one of the 4 biased periods when moving from the true label to the biased one -- a clean, consistent, fully attributable signal confirming the bias is detectable via outcome-aware metrics (Equalized Odds), even though a naive demographic-parity-only audit would completely miss it.

## 3. Fairness-accuracy trade-off report

Runs `src/fairness/fairness_accuracy_tradeoff.py` on **age_band** (the attribute with a real, substantial natural disparity -- DPD=0.057, EqualizedOddsDiff=0.066 -- unlike CODE_GENDER, which is already close to parity on real data and has nothing meaningful to trade off; that near-null result is saved separately as `data/processed/phase4_fairness_accuracy_tradeoff_CODE_GENDER.csv`).

In [5]:
%run ../src/fairness/fairness_accuracy_tradeoff.py

baseline         approval_rate=0.9697  profit=18,075,365,325  EL_approved=7,083,511,999  DPD=0.0565  EOD=0.0472  EqOdds=0.0660
equalize_dp      approval_rate=0.9707  profit=18,051,133,297  EL_approved=7,100,327,059  DPD=0.0021  EOD=0.0039  EqOdds=0.0417
equalize_eo      approval_rate=0.9708  profit=18,057,592,793  EL_approved=7,101,606,205  DPD=0.0055  EOD=0.0010  EqOdds=0.0471
blunt_stricter   approval_rate=0.8829  profit=17,641,314,593  EL_approved=5,683,887,111  DPD=0.1722  EOD=0.1542  EqOdds=0.1542

Saved: D:\finance\credit-risk-fyp\data\processed\phase4_fairness_accuracy_tradeoff_age_band.csv

equalize_dp vs baseline: profit cost = 24,232,028 (0.13%), DPD gain = 0.0544, EOD gain = 0.0433

equalize_eo vs baseline: profit cost = 17,772,531 (0.10%), DPD gain = 0.0510, EOD gain = 0.0462

blunt_stricter vs baseline: profit cost = 434,050,732 (2.40%), DPD gain = -0.1157, EOD gain = -0.1070


In [6]:
result

,auc,approval_rate,profit,el_approved,DPD,EOD,EqualizedOddsDiff
strategy,,,,,,,
baseline,0.308158,0.969660,1.807537e+10,7.083512e+09,0.056493,0.047180,0.065980
equalize_dp,0.308158,0.970707,1.805113e+10,7.100327e+09,0.002108,0.003925,0.041733
equalize_eo,0.308158,0.970759,1.805759e+10,7.101606e+09,0.005489,0.001026,0.047052
blunt_stricter,0.308158,0.882895,1.764131e+10,5.683887e+09,0.172206,0.154208,0.154208


**Result**: targeted, group-specific threshold mitigation is remarkably cheap here -- `equalize_dp` closes ~96% of the DPD gap and `equalize_eo` closes ~99% of the EOD gap for a profit cost under 0.15% either way. `blunt_stricter` (a naive uniform stricter threshold, no group-awareness) is the clear cautionary result: it costs 2.40% of profit -- 15-20x more than the targeted strategies -- while making DPD and EOD *worse* than baseline, not better. That's a genuinely useful, concrete finding for the dissertation's fairness-accuracy trade-off section: blunt mitigation isn't just expensive, it can be actively counterproductive.